In [1]:
!pip install transformers datasets accelerate scikit-learn -q

In [4]:
from datasets import load_dataset
import pandas as pd


dataset_names = [
    "UniversalCEFR/elg_cefr_en",
    "UniversalCEFR/cambridge_exams_en",
    "UniversalCEFR/cefr_asag_en",
    "UniversalCEFR/icle500_en",
]

 # Collapse sub-levels (A2+ → A2, etc.)
LABEL_MAP = {
    "A1": "A1", "A1+": "A1",
    "A2": "A2", "A2+": "A2",
    "B1": "B1", "B1+": "B1",
    "B2": "B2", "B2+": "B2",
    "C1": "C1", "C1+": "C1",
    "C2": "C2",
}

frames = []
for name in dataset_names:
    ds = load_dataset(name)
    df = ds["train"].to_pandas()
    df = df[df["format"] == "document-level"].copy()
    df["cefr_level"] = df["cefr_level"].map(LABEL_MAP)
    df = df.dropna(subset=["cefr_level"])
    df = df[["text", "cefr_level"]]
    print(f"{name}: {len(df)} docs")
    frames.append(df)

# Kaggle CEFR dataset (already document-level)
df_kaggle = pd.read_csv("/content/cefr_leveled_texts.csv")
df_kaggle = df_kaggle.rename(columns={"label": "cefr_level"})
df_kaggle["cefr_level"] = df_kaggle["cefr_level"].map(LABEL_MAP)
df_kaggle = df_kaggle.dropna(subset=["cefr_level"])
df_kaggle = df_kaggle[["text", "cefr_level"]]
print(f"Kaggle CSV: {len(df_kaggle)} docs")
frames.append(df_kaggle)

df = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df)} documents")
print(df["cefr_level"].value_counts().sort_index())

README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

elg-cefr-en.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/712 [00:00<?, ? examples/s]

UniversalCEFR/elg_cefr_en: 712 docs


README.md:   0%|          | 0.00/754 [00:00<?, ?B/s]

...

UniversalCEFR/cefr_asag_en: 299 docs


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

icle500.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/495 [00:00<?, ? examples/s]

UniversalCEFR/icle500_en: 488 docs
Kaggle CSV: 1494 docs

Total: 3324 documents
cefr_level
A1    331
A2    537
B1    698
B2    823
C1    526
C2    409
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# 80/20 stratified split
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["cefr_level"]
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")


label2id = {label: i for i, label in enumerate(sorted(df["cefr_level"].unique()))}
id2label = {i: label for label, i in label2id.items()}
print(f"Label mapping: {label2id}")


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512,
)
test_encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=512,
)

train_labels = [label2id[l] for l in train_df["cefr_level"]]
test_labels = [label2id[l] for l in test_df["cefr_level"]]

print(f"Tokenized. Example input length: {len(train_encodings['input_ids'][0])} tokens")

Train: 2659, Test: 665
Label mapping: {'A1': 0, 'A2': 1, 'B1': 2, 'B2': 3, 'C1': 4, 'C2': 5}


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenized. Example input length: 512 tokens


In [6]:
import torch
from transformers import AutoModelForSequenceClassification

# PyTorch Dataset wrapper for HuggingFace encodings
class CEFRDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CEFRDataset(train_encodings, train_labels)
test_dataset = CEFRDataset(test_encodings, test_labels)

# DistilBERT + 6-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded. Parameters: 66,958,086


In [7]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./cefr-distilbert",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.166978,1.075166,0.568421,0.530823
2,0.980207,0.931749,0.604511,0.610103
3,0.769755,0.893283,0.646617,0.663113
4,0.644239,0.906829,0.651128,0.669882
5,0.557172,0.879991,0.679699,0.698678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=835, training_loss=0.8718649378793683, metrics={'train_runtime': 773.5296, 'train_samples_per_second': 17.187, 'train_steps_per_second': 1.079, 'total_flos': 1761279695861760.0, 'train_loss': 0.8718649378793683, 'epoch': 5.0})

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = np.array(test_labels)

labels = [id2label[i] for i in range(len(id2label))]

print(classification_report(y_true, y_pred, target_names=labels))

print("\nConfusion matrix:")
print(labels)
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

          A1       0.85      0.83      0.84        66
          A2       0.73      0.86      0.79       107
          B1       0.71      0.57      0.63       140
          B2       0.61      0.65      0.63       165
          C1       0.53      0.56      0.54       105
          C2       0.80      0.72      0.76        82

    accuracy                           0.68       665
   macro avg       0.70      0.70      0.70       665
weighted avg       0.68      0.68      0.68       665


Confusion matrix:
['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
[[ 55  11   0   0   0   0]
 [  8  92   7   0   0   0]
 [  2  19  80  33   5   1]
 [  0   2  20 107  31   5]
 [  0   2   5  30  59   9]
 [  0   0   0   6  17  59]]


In [9]:
import json

# Save model + tokenizer to a directory
save_path = "./cefr-distilbert-final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Save label mappings
with open(f"{save_path}/label_config.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

# Zip for download
!zip -r cefr-distilbert-final.zip cefr-distilbert-final/

print(f"Saved to {save_path}/ and zipped for download")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: cefr-distilbert-final/ (stored 0%)
  adding: cefr-distilbert-final/training_args.bin (deflated 53%)
  adding: cefr-distilbert-final/tokenizer.json (deflated 71%)
  adding: cefr-distilbert-final/model.safetensors (deflated 8%)
  adding: cefr-distilbert-final/config.json (deflated 52%)
  adding: cefr-distilbert-final/tokenizer_config.json (deflated 42%)
  adding: cefr-distilbert-final/label_config.json (deflated 53%)
Saved to ./cefr-distilbert-final/ and zipped for download
